<a href="https://colab.research.google.com/github/bmanibharathibe/trendpulse-manibharathi/blob/main/Data_Processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import requests
import json
import os
import time
from datetime import datetime

headers = {"User-Agent": "TrendPulse/1.0"}

TOP_STORIES_URL = "https://hacker-news.firebaseio.com/v0/topstories.json"
ITEM_URL = "https://hacker-news.firebaseio.com/v0/item/{}.json"

categories = {
    "technology": ["ai","software","tech","code","computer","data","cloud","api","gpu","llm"],
    "worldnews": ["war","government","country","president","election","climate","attack","global"],
    "sports": ["nfl","nba","fifa","sport","game","team","player","league","championship"],
    "science": ["research","study","space","physics","biology","discovery","nasa","genome"],
    "entertainment": ["movie","film","music","netflix","game","book","show","award","streaming"]
}

collected_stories = []
category_counts = {cat: 0 for cat in categories}

# Fetch top story IDs
try:
    response = requests.get(TOP_STORIES_URL, headers=headers)
    story_ids = response.json()
except:
    print("Error fetching top stories")
    story_ids = []

# Function to detect category
def detect_category(title):
    title = title.lower()
    for cat, words in categories.items():
        for word in words:
            if word in title:
                return cat
    return None

# Loop through categories
for category in categories:

    print(f"Collecting stories for {category}")

    for story_id in story_ids:

        if category_counts[category] >= 35:
            break

        try:
            r = requests.get(ITEM_URL.format(story_id), headers=headers)
            story = r.json()
        except:
            print("Request failed for", story_id)
            continue

        if not story or "title" not in story:
            continue

        detected = detect_category(story["title"])

        if detected != category:
            continue

        story_data = {
            "post_id": story.get("id"),
            "title": story.get("title"),
            "category": category,
            "score": story.get("score", 0),
            "num_comments": story.get("descendants", 0),
            "author": story.get("by"),
            "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }

        collected_stories.append(story_data)
        category_counts[category] += 1

    # sleep after each category
    time.sleep(2)

# Create data folder
if not os.path.exists("data"):
    os.makedirs("data")

filename = f"data/trends_{datetime.now().strftime('%Y%m%d')}.json"

with open(filename, "w") as f:
    json.dump(collected_stories, f, indent=4)

print(f"Collected {len(collected_stories)} stories. Saved to {filename}")

Collected 112 stories. Saved to data/trends_20260408.json


In [5]:
import pandas as pd
import os
import json
import glob

# -------------------------------
# Step 1 — Load the JSON file
# -------------------------------

# Find the latest JSON file in the data folder
files = glob.glob("data/trends_*.json")

if not files:
    print("No JSON file found in data/ folder. Please run task1_data_collection.py first.")
    exit()

latest_file = max(files)

# Load JSON into pandas DataFrame
df = pd.read_json(latest_file)

print(f"Loaded {len(df)} stories from {latest_file}")

# -------------------------------
# Step 2 — Clean the data
# -------------------------------

# Remove duplicate stories based on post_id
before = len(df)
df = df.drop_duplicates(subset="post_id")
print(f"After removing duplicates: {len(df)}")

# Remove rows with missing values in key fields
df = df.dropna(subset=["post_id", "title", "score"])
print(f"After removing nulls: {len(df)}")

# Convert score and num_comments to integers
df["score"] = df["score"].astype(int)
df["num_comments"] = df["num_comments"].astype(int)

# Remove low-quality stories (score < 5)
df = df[df["score"] >= 5]
print(f"After removing low scores: {len(df)}")

# Remove extra whitespace from titles
df["title"] = df["title"].str.strip()

# -------------------------------
# Step 3 — Save cleaned CSV
# -------------------------------

output_file = "data/trends_clean.csv"
df.to_csv(output_file, index=False)

print(f"\nSaved {len(df)} rows to {output_file}")

# -------------------------------
# Summary: stories per category
# -------------------------------

print("\nStories per category:")

category_summary = df["category"].value_counts()

for category, count in category_summary.items():
    print(f"  {category:<15} {count}")

Loaded 112 stories from data/trends_20260408.json
After removing duplicates: 112
After removing nulls: 112
After removing low scores: 108

Saved 108 rows to data/trends_clean.csv

Stories per category:
  technology      34
  entertainment   33
  worldnews       21
  sports          14
  science         6
